# 02. Path in a graph

**Tier:** Deep. **Why you'd use this:** Rooms are nodes and doors are undirected edges. You want the fewest doors to walk from the entrance to a target room. Breadth-first search finds that distance.

**After this lesson you can:** build an adjacency list; walk a queue level by level; mark nodes when you enqueue them.
About 25 minutes. **Uses:** [deque](../../course/08-collections-module/03-deque-lesson.ipynb), [Dicts](../../course/04-collections/03-dicts-lesson.ipynb)


## Quick review

1. What does `popleft` do on a `deque`?
2. Why store neighbors in a list per node before searching?

<details><summary>Answers</summary>

Removes and returns the leftmost value — FIFO when you append on the right. An adjacency list turns edge scans into constant-time neighbor iteration. Skim [deque](../../course/08-collections-module/03-deque-lesson.ipynb) if the queue API is new.
</details>


## Recognize it

You have nodes and unweighted edges and need the shortest path length (fewest edges) between two nodes. The prompt says "shortest path in an unweighted graph", "fewest stops", or "minimum edges".

Signals: BFS with a queue; first time you reach the target is optimal. A look-alike that is not this pattern: weighted edges — then Dijkstra (or another priority search) replaces the plain queue.


## Why it works this way

In an unweighted graph every edge has the same cost. Expanding nodes in the order you first discover them means the first time you reach the target you have used the fewest edges. That is breadth-first search: a queue holds the frontier, and you append neighbors at distance `d + 1` only after finishing distance `d`.

Mark a node when you enqueue it, not when you dequeue it. Otherwise two parents can both push the same neighbor and you waste work (and can break the "first visit is shortest" story if you also store distances lazily). Building an adjacency list from the edge list is a linear preprocess: each undirected edge becomes two directed links.

Depth-first search can find *a* path but not the shortest one without extra machinery. Dijkstra generalizes BFS when edge weights are positive and unequal. Bidirectional BFS is an optimization when both ends are known and the branching factor is high.

Time is proportional to nodes plus edges. Extra memory is the adjacency lists, the queue, and the seen set.
Grid shortest path is the same BFS with four (or eight) neighbor offsets and a 2-D seen structure. 0-1 BFS uses a deque and pushes weight-0 edges to the left — a close cousin when most edges are free. Topological order on directed acyclic graphs uses indegrees and a queue but answers a different question: a valid processing order, not a distance.

If the graph might be disconnected, unreachable targets return `-1` (or another sentinel). Always handle `source == target` as distance zero before worrying about edges. Building the adjacency list with sets instead of lists deduplicates parallel edges when the input might repeat them; lists are fine when the prompt guarantees simple edges.


## When to use it

Use BFS for fewest edges (or level order) on unweighted graphs and grids. Prefer DFS or recursion when you only need connectivity, not distance. Prefer Dijkstra when edges have different positive weights. Prefer Union-Find when you only merge components and never need a path.
 Early exit when the target is enqueued (instead of dequeued) also works if you store steps beside the node; both styles appear in production code.


## Brute force, and why it's slow

Enumerate all simple paths with backtracking and keep the shortest. The path count can grow exponentially.


In [1]:
def fewest_edges_slow(n, edges, source, target):
    graph = [[] for _ in range(n)]
    for left, right in edges:
        graph[left].append(right)
        graph[right].append(left)
    best = None
    paths = 0

    def walk(node, steps, seen):
        nonlocal best, paths
        paths += 1
        if node == target:
            best = steps if best is None else min(best, steps)
            return
        for nxt in graph[node]:
            if nxt not in seen:
                walk(nxt, steps + 1, seen | {nxt})

    walk(source, 0, {source})
    return (-1 if best is None else best), paths


print(fewest_edges_slow(4, [[0, 1], [1, 2], [2, 3], [0, 3]], 0, 3))
print(fewest_edges_slow(3, [[0, 1]], 0, 2))


(1, 5)
(-1, 2)


## Key insight

BFS reaches each node for the first time along a shortest path in an unweighted graph. Store steps in the queue and stop when the target is dequeued (or enqueued).


## Worked example

Return the fewest edges from `source` to `target` in an undirected unweighted graph, or `-1` when unreachable.


In [2]:
from collections import deque


def fewest_edges(n: int, edges: list[list[int]], source: int, target: int) -> int:
    # 1. Undirected adjacency lists.
    graph = [[] for _ in range(n)]
    for left, right in edges:
        graph[left].append(right)
        graph[right].append(left)
    # 2. Mark on enqueue; first visit is shortest.
    seen = {source}
    queue = deque([(source, 0)])
    while queue:
        node, steps = queue.popleft()
        if node == target:
            return steps
        for nxt in graph[node]:
            if nxt not in seen:
                seen.add(nxt)
                queue.append((nxt, steps + 1))
    return -1


print(fewest_edges(3, [[0, 1], [1, 2]], 0, 2))
print(fewest_edges(3, [[0, 1]], 0, 2))
print(fewest_edges(1, [], 0, 0))


2
-1
0


## Trace

`n=3`, edges `0—1—2`, source `0`, target `2`:

| dequeue | steps | enqueue |
|---|---|---|
| 0 | 0 | 1 |
| 1 | 1 | 2 |
| 2 | 2 | — → return **2** |


## Cost

Each node and edge is processed a constant number of times: time grows with `n + len(edges)`. Extra memory is the graph, queue, and seen set. Exhaustive path search can explode with the number of routes.

Adjacency lists use memory proportional to edges. An adjacency matrix would use `n²` space and help only when the graph is dense and you need O(1) edge queries. Sparse door-and-room graphs stay with lists.


## Variants

Not solved here: [Number of Islands](../../challenges/problems/12-number-of-islands.ipynb), [Course Schedule](../../challenges/problems/33-course-schedule.ipynb), [Clone Graph](../../challenges/problems/48-clone-graph.ipynb).


## Common mistake

This bug is silent on short paths and wasteful always. The code marks a node only when it is dequeued, so the same neighbor can be enqueued many times from different parents. On a dense graph the queue balloons. For the tiny example the distance may still look right, which hides the bug.


In [3]:

def fewest_edges_wrong(n, edges, source, target):
    graph = [[] for _ in range(n)]
    for left, right in edges:
        graph[left].append(right)
        graph[right].append(left)
    seen = set()
    queue = deque([(source, 0)])
    enqueued = 0
    while queue:
        node, steps = queue.popleft()
        if node in seen:
            continue
        seen.add(node)  # mark on dequeue — too late
        if node == target:
            return steps, enqueued
        for nxt in graph[node]:
            queue.append((nxt, steps + 1))
            enqueued += 1
    return -1, enqueued


print(fewest_edges_wrong(4, [[0, 1], [0, 2], [1, 3], [2, 3]], 0, 3))


(2, 6)


The fix adds to `seen` when pushing onto the queue so each node enters at most once. Run `fewest_edges(3, [[0, 1], [1, 2]], 0, 2)` and you get `2`.


In [4]:
print(fewest_edges(3, [[0, 1], [1, 2]], 0, 2))


2


## Python idioms

`collections.deque` gives amortized O(1) `popleft`. A list used with `pop(0)` would shift on every dequeue and quietly make BFS quadratic.


In [5]:

def reachable(n: int, edges: list[list[int]], source: int) -> set[int]:
    graph = [[] for _ in range(n)]
    for left, right in edges:
        graph[left].append(right)
        graph[right].append(left)
    seen = {source}
    queue = deque([source])
    while queue:
        node = queue.popleft()
        for nxt in graph[node]:
            if nxt not in seen:
                seen.add(nxt)
                queue.append(nxt)
    return seen


print(sorted(reachable(3, [[0, 1], [1, 2]], 0)))


[0, 1, 2]


## Recap

- BFS finds fewest edges in an unweighted graph.
- Mark on enqueue so each node is processed once.
- Use a `deque`, not `list.pop(0)`.

Next: [practice](02-path-practice.ipynb).
